In [ ]:
import sys
sys.path.append("/exp/sbnd/data/users/lnguyen/cafpyana_pi0")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from   dataclasses import replace
import cafpybara.analyses.nuecc as nue
from makedf.geniesyst import regen_systematics, ar23p_genie_systematics

%load_ext autoreload
%autoreload 2

## load dataframes

In [ ]:
dfs_dir = "/exp/sbnd/data/users/lynnt/xsection/samples/MCP2025B_v10_06_00_09/dfs_nu26/"

mcbnb_orig_df, mcbnb_pot, mcbnb_ngen = nue.load_mc(
    dfs_dir + "mc_ar23.df",
    keys=['nuecc', 'hdr'],
    chunk_splits=5,
    cuts=nue.SIDEBAND_CUTS,
)
mclow_df,  mclow_pot,  mclow_ngen   = nue.load_mc(
    dfs_dir + "mc_lowenergy.df",
    keys=['nuecc', 'hdr', 'histpotdf'],
    cuts=nue.SIDEBAND_CUTS,
)
dtbnb_df,  dtbnb_pot,  dtbnb_ngates = nue.load_data(
    dfs_dir + "data_dev.df",
    keys=['nuecc', 'hdr'],
    onbeam=True,
    cuts=nue.SIDEBAND_CUTS,
)
dtoff_df,  dtoff_pot,  dtoff_ngates = nue.load_data(
    dfs_dir + "offbeamlight.df",
    keys=['nuecc', 'hdr'],
    onbeam=False,
    cuts=nue.SIDEBAND_CUTS,
)

In [ ]:
systs_to_drop = [
    'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b1',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b2',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b3',
    'ZExpPCAWeighter_SBNNuSyst_multisigma_MvA_ZExp_b4',
]
systs_list = regen_systematics + ar23p_genie_systematics
for dropcol in systs_to_drop:
    if dropcol in systs_list:
        systs_list.remove(dropcol)
    if dropcol in list(zip(*list(mcbnb_orig_df.columns)))[2]:
        mcbnb_orig_df.drop(dropcol, axis=1, level=2, inplace=True)

mcbnb_df = nue.make_multiverse_weights(
    evtdf=nue.ensure_lexsorted(mcbnb_orig_df, axis=1),
    knob_list=systs_list,
    evt_prefix=('slc', 'truth'),
    drop_originals=True,
)

In [ ]:
ongates_per_pot  = dtbnb_ngates / dtbnb_pot
f                = 0.0725
noffbeamscale_mc = ((1 - f) * (ongates_per_pot * mcbnb_pot)) / dtoff_ngates
scale            = dtbnb_pot / mcbnb_pot

mcbnb_df[('weights_mc', '', '', '', '', '')] = 1.0
mclow_df[('weights_mc', '', '', '', '', '')] = mcbnb_pot / mclow_pot
dtoff_df[('weights_mc', '', '', '', '', '')] = noffbeamscale_mc

In [ ]:
mcstat_cols      = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample', 'file_idx']
mcmc_index_names = ['__ntuple', 'entry', 'file_idx', 'rec.slc..index', 'sample']

mcmc_sideband_df = pd.concat([
    nue.mcstat(mcbnb_df.assign(sample=0).reset_index(), cols=mcstat_cols),
    nue.mcstat(mclow_df.assign(sample=2).reset_index(), cols=mcstat_cols),
    dtoff_df.assign(sample=3).reset_index(),
])

mcmc_sideband_df.set_index(mcmc_index_names, inplace=True)
data_sideband_df = dtbnb_df.copy()

## configure systematics and plotting

In [ ]:
sideband_detvar_dict = nue.load_detvar_dict(
    dfs_dir + "detvars/detvars_sideband.h5",
    preprocess_fn=nue.add_pi0,
)

sideband_systs_cfg = nue.systs_input(
    mcbnb_pot,
    mcbnb_ngen=mcbnb_ngen,
    detvar_dict=sideband_detvar_dict,
    # cuts= must match the cuts mcmc_sideband_df was loaded with above
    # (SIDEBAND_CUTS) -- systs_input()'s own cuts=None default is
    # DEFAULT_CUTS (the signal region), not this, so this must stay
    # explicit. See README's "Common mistakes" section for why.
    cuts=nue.SIDEBAND_CUTS,
)

sideband_plots_cfg = nue.PlottingConfig(
    scale=scale,
    ylabel=f"Events [{dtbnb_pot:.2e} POT]",
    legend_kwargs={'loc': 'upper right', 'ncol': 3, 'fontsize': 9},
    systs=sideband_systs_cfg,
    percents=True,
    data_first=False,
    title=" ",
)

SIDEBAND_TRK_CUTS   = nue.SIDEBAND_CUTS + [nue.CutSpec("trk", fn=lambda df: ~df.primtrk.trk.len.isna())]
sideband_trk_systs_cfg = replace(sideband_systs_cfg, cuts=SIDEBAND_TRK_CUTS)
sideband_trk_plots_cfg = replace(sideband_plots_cfg, systs=sideband_trk_systs_cfg)

## sideband plots

In [ ]:
output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df, data_df=data_sideband_df,
    var=nue.electron_direction().var_evt_reco_col,
    bins=nue.electron_direction().bins,
    bin_labels=nue.electron_direction().bin_labels,
    xlabel=r"Leading Shower Direction, $\cos\theta$",
    legend_kwargs={'loc': 'upper left', 'ncol': 2, 'fontsize': 9},
    config=sideband_plots_cfg)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df, data_df=data_sideband_df,
    var=nue.electron_energy().var_evt_reco_col,
    bins=nue.electron_energy().bins,
    bin_labels=nue.electron_energy().bin_labels,
    xlabel="Leading Shower Energy [GeV]",
    legend_kwargs={'loc': 'upper right', 'ncol': 2, 'fontsize': 9},
    config=sideband_plots_cfg)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df, data_df=data_sideband_df,
    var=('primshw', 'shw', 'dir', 'phi'),
    xlabel=r"Leading Shower $\phi$ [degrees]",
    pdg=True, pdg_col=('primshw', 'shw', 'truth', 'p', 'pdg'),
    bins=np.linspace(-180, 180, 10),
    config=sideband_plots_cfg)
output[1].set_ylim(output[1].get_ylim()[0], output[1].get_ylim()[1] * 1.75)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df, data_df=data_sideband_df,
    var=('primshw', 'shw', 'open_angle'),
    xlabel="Leading Shower Opening Angle [rad]",
    legend_kwargs={'loc': 'upper right', 'ncol': 2, 'fontsize': 9},
    bins=np.linspace(0, 0.5, 11),
    config=sideband_plots_cfg)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df, data_df=data_sideband_df,
    var=('slc', 'nshw'),
    legend_kwargs={'loc': 'upper right', 'ncol': 3, 'fontsize': 9},
    bins=np.arange(1, 6),
    config=sideband_plots_cfg)
output[1].set_ylim(output[1].get_ylim()[0], output[1].get_ylim()[1] * 1.5)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df, data_df=data_sideband_df,
    var=('slc', 'ntrk'),
    legend_kwargs={'loc': 'upper right', 'ncol': 3, 'fontsize': 9},
    bins=np.arange(0, 6),
    config=sideband_plots_cfg)
output[1].set_ylim(output[1].get_ylim()[0], output[1].get_ylim()[1] * 1.5)
plt.show()

In [ ]:
has_trk      = mcmc_sideband_df.primtrk.trk.len.notna()
has_trk_data = data_sideband_df.primtrk.trk.len.notna()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df[has_trk], data_df=data_sideband_df[has_trk_data],
    var=('primtrk', 'trk', 'dir', 'phi'),
    xlabel=r"Leading Track $\phi$ [degrees]",
    pdg=True, pdg_col=('primtrk', 'trk', 'truth', 'p', 'pdg'),
    bins=np.linspace(-180, 180, 10),
    config=sideband_trk_plots_cfg)
output[1].set_ylim(output[1].get_ylim()[0], output[1].get_ylim()[1] * 1.75)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_sideband_df[has_trk], data_df=data_sideband_df[has_trk_data],
    var=('primtrk', 'trk', 'len'),
    xlabel="Leading Track Length [cm]",
    pdg=True, pdg_col=('primtrk', 'trk', 'truth', 'p', 'pdg'),
    bins=np.linspace(0, 300, 11),
    legend_kwargs={'loc': 'upper right', 'ncol': 3, 'fontsize': 9},
    config=sideband_trk_plots_cfg)
output[1].set_ylim(output[1].get_ylim()[0], output[1].get_ylim()[1] * 1.25)
plt.show()

## pi0 sideband

In [ ]:
SIDEBAND_PI0_CUTS = nue.SIDEBAND_CUTS + [
    nue.CutSpec("secshw", fn=lambda df: df.secshw.shw.reco_energy > 0.02),
]
mcmc_pi0_df = nue.add_pi0(nue.select(mcmc_sideband_df, cuts=SIDEBAND_PI0_CUTS))
data_pi0_df = nue.add_pi0(nue.select(data_sideband_df, cuts=SIDEBAND_PI0_CUTS))

pi0_systs_cfg = nue.systs_input(
    mcbnb_pot,
    mcbnb_ngen=mcbnb_ngen,
    detvar_dict=sideband_detvar_dict,
    cuts=SIDEBAND_PI0_CUTS,
)
pi0_plots_cfg = nue.PlottingConfig(
    scale=scale,
    ylabel=f"Events [{dtbnb_pot:.2e} POT]",
    legend_kwargs={'loc': 'upper right', 'ncol': 2, 'fontsize': 9},
    systs=pi0_systs_cfg,
    percents=True,
    data_first=False,
    title=" ",
)

In [ ]:
output = nue.plot_mc_data(
    mc_df=mcmc_pi0_df, data_df=data_pi0_df,
    var=('pi0', 'mass'),
    bins=np.linspace(0, 1.0, 11),
    xlabel=r"Reconstructed $\pi^0$ Mass [GeV]",
    config=pi0_plots_cfg)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_pi0_df, data_df=data_pi0_df,
    var=('pi0', 'p', 'totp'),
    bins=np.linspace(0.5, 2, 11),
    xlabel=r"Reconstructed $\pi^0$ Momentum [GeV/c]",
    config=pi0_plots_cfg)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_pi0_df, data_df=data_pi0_df,
    var=('pi0', 'openangle'),
    bins=np.linspace(0, 150, 11),
    xlabel="Angle Between Leading/Subleading Showers [Degrees]",
    config=pi0_plots_cfg)
output[1].set_ylim(output[1].get_ylim()[0], output[1].get_ylim()[1] * 1.25)
plt.show()

output = nue.plot_mc_data(
    mc_df=mcmc_pi0_df, data_df=data_pi0_df,
    var=('pi0', 'dir', 'z'),
    bins=nue.electron_direction().bins,
    bin_labels=nue.electron_direction().bin_labels,
    legend_kwargs={'loc': 'upper left', 'ncol': 2, 'fontsize': 9},
    xlabel=r"$\pi^0$ Direction, $\cos\theta_{\pi^0}$",
    config=pi0_plots_cfg)
plt.show()

## systematics breakdown

In [ ]:
sideband_energy_output = nue.get_total_cov(
    mcmc_sideband_df,
    reco_var=nue.electron_energy().var_evt_reco_col,
    bins=nue.electron_energy().bins,
    **sideband_systs_cfg.to_kwargs(),
)
sideband_direct_output = nue.get_total_cov(
    mcmc_sideband_df,
    reco_var=nue.electron_direction().var_evt_reco_col,
    bins=nue.electron_direction().bins,
    **sideband_systs_cfg.to_kwargs(),
)

In [ ]:
_dir_var = nue.electron_direction()
_ene_var = nue.electron_energy()
syst_vars = [
    (sideband_energy_output, _ene_var.bins, "Leading Shower Energy [GeV]",             _ene_var.bin_labels),
    (sideband_direct_output, _dir_var.bins, r"Leading Shower Direction, $\cos\theta$", _dir_var.bin_labels),
]

fig, axes, cats, cat_sums = nue.plot_syst_category_breakdown(
    syst_vars=syst_vars,
    category_dict=nue.category_dict_control,
    region_label="Sideband (Control Region)",
    show_cv=True,
)
plt.show()

for category in ['DetVar', 'Flux', 'Geant4', 'GENIE']:
    fig, axes = nue.plot_syst_breakdown(
        syst_vars=syst_vars,
        category=category,
        category_dict=nue.category_dict_control,
        region_label="Sideband (Control Region)",
    )
    plt.show()